# ConvNeXt-B full fine-tune on Kaggle (GPU T4 ×2)

Same recipe as `ViT_finetune_colab.ipynb` (private 77.50 with the fold-0 ViT), with a CNN backbone so the two models make different mistakes and ensemble well.

- Backbone `convnext_base` / `ConvNeXt_Base_Weights.IMAGENET1K_V1`, fully fine-tuned (layer-wise LR decay over 13 depth groups, AMP fp16, gradient checkpointing per block) + the same temporal Transformer head.
- Per-frame feature = [avg-pool, max-pool] of the final (LayerNorm'd) 7×7 map, 2048-d: max-pool keeps small, local evidence such as hand shape.
- Full 224×224 frames, 16 frames, blank padding frames dropped, strong clip-consistent augmentation that never crops the borders, one random file per original video each epoch.
- Uses both T4s with `nn.DataParallel` when the accelerator is **GPU T4 x2**.

**Kaggle setup**
1. Settings → Accelerator: **GPU T4 x2**; Internet: **On** (pretrained weights; dataset download if not attached).
2. Data: either *Add Input* a Kaggle dataset containing `label_mapping.pkl`, `train/`, `public_test/`, `private_test/` (found automatically anywhere under `/kaggle/input`), or leave it out and the notebook downloads BTC's `dataset.zip` from Google Drive.
3. **Save Version → Save & Run All (Commit)**: runs headless (up to 12 h), so closing the browser does not kill it. Results land in the *Output* tab: `cv_submission_<run>.zip`, logits `.npy`, `history.json`.
4. Resume after a failed/stopped version: add that version's output as an input; `<run>_last.pth` is picked up from `/kaggle/input` automatically.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
import os, sys, re, glob, time, math, json, pickle, random, zipfile, unicodedata

ON_KAGGLE = os.path.isdir('/kaggle/input')
DATASET_URL = 'https://drive.google.com/file/d/1Bg-PByokbKDAftvU6LhrHgXesAPpZqd5/view'   # BTC's CV/dataset.zip (fallback when no Kaggle input has the data)


def download_drive(url, dst):
    import requests
    from tqdm.auto import tqdm
    m = re.search(r'/d/([\w-]+)|id=([\w-]+)', url)
    file_id = (m.group(1) or m.group(2)) if m else url.strip()
    with requests.get(f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t', stream=True, timeout=60) as r:
        r.raise_for_status()
        assert 'text/html' not in r.headers.get('content-type', ''), f'Drive returned a web page, not the file (Internet off, link not public or quota hit):\n{r.text[:500]}'
        with open(dst + '.part', 'wb') as f, tqdm(total=int(r.headers.get('content-length', 0)), unit='B', unit_scale=True, desc=os.path.basename(dst)) as bar:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk); bar.update(len(chunk))
    os.rename(dst + '.part', dst)


if ON_KAGGLE:
    OUT_DIR = '/kaggle/working'
    found = sorted(glob.glob('/kaggle/input/**/label_mapping.pkl', recursive=True))
    found = [os.path.dirname(p) for p in found if os.path.isdir(os.path.join(os.path.dirname(p), 'train'))]
    if found:
        DATA = found[0]
    else:
        work = '/tmp/sl'   # scratch disk: not saved into the version output
        os.makedirs(work, exist_ok=True)
        DATA = f'{work}/dataset'
        if not os.path.exists(f'{DATA}/label_mapping.pkl'):
            zip_path = f'{work}/dataset.zip'
            if not (os.path.exists(zip_path) and zipfile.is_zipfile(zip_path)):
                download_drive(DATASET_URL, zip_path)
            t0 = time.time()
            with zipfile.ZipFile(zip_path) as zf:   # zipfile (not `unzip`) keeps the Vietnamese folder names intact
                zf.extractall(work)
            os.remove(zip_path)
            print(f'extracted in {time.time() - t0:.0f}s')
else:
    DATA, OUT_DIR = 'dataset', 'work'
os.makedirs(OUT_DIR, exist_ok=True)
print('data:', DATA, '| outputs ->', OUT_DIR, '| cpus:', os.cpu_count())

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)   # DataLoader workers do the parallelism

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
N_GPU = torch.cuda.device_count() if DEVICE == 'cuda' else 0
# bf16 only on Ampere+; T4 (sm75) / P100 (sm60) -> fp16 + GradScaler
AMP_DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
if DEVICE == 'cuda':
    print(' + '.join(torch.cuda.get_device_name(i) for i in range(N_GPU)), '| amp', AMP_DTYPE)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True   # fixed 224x224 input

NUM_CLASSES = 100
N_FOLDS = 5
T = 16
SEED = 42
WEIGHTS = ConvNeXt_Base_Weights.IMAGENET1K_V1

VAL_FOLD = 0          # int -> train on the other folds and report val Macro-F1; None -> train on everything (final model)
EPOCHS = 20
BATCH = 8             # videos per step (x T frames), split across the GPUs
LR_BACKBONE = 5e-5    # top depth group; lower groups get LR_BACKBONE * LAYER_DECAY**depth
LAYER_DECAY = 0.8
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.05
DROP_PATH = 0.2       # stochastic depth (torchvision's default for convnext_base is 0.5, too strong for 2.8k videos)
LABEL_SMOOTH = 0.1
GRAD_CKPT = True      # recompute ConvNeXt blocks in backward: much less memory, ~30% slower
TIME_BUDGET_H = 11.0  # Kaggle kills a session at 12 h: stop training early enough to still predict
NUM_WORKERS = os.cpu_count() or 2
RUN_NAME = f'convnextb_ft_{"full" if VAL_FOLD is None else f"fold{VAL_FOLD}"}'
T_START = time.time()

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()
print(RUN_NAME, '| seed', SEED)

## Data & folds (same grouped, exactly stratified split as `ViT.ipynb`)

In [ ]:
nfc = lambda s: unicodedata.normalize('NFC', s)
with open(f'{DATA}/label_mapping.pkl', 'rb') as f:
    raw_mapping = pickle.load(f)
label_to_idx = {nfc(k): v for k, v in raw_mapping.items()}
idx_to_label = {v: k for k, v in raw_mapping.items()}   # organiser's exact strings for the CSV

rows = []
for label in sorted(os.listdir(f'{DATA}/train')):
    d = f'{DATA}/train/{label}'
    if not os.path.isdir(d):
        continue
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.mp4'):
            stem = fn[:-4]
            rows.append({'path': f'{d}/{fn}', 'label_idx': label_to_idx[nfc(label)],
                         'group': re.sub(r'_\d+$', '', stem), 'is_orig': re.search(r'_\d+$', stem) is None})
df = pd.DataFrame(rows)

rng = np.random.default_rng(SEED)
group_fold, offset = {}, 0
for _, g in df.groupby('label_idx'):
    groups = g.group.unique()
    rng.shuffle(groups)
    for i, grp in enumerate(groups):
        group_fold[grp] = (i + offset) % N_FOLDS
    offset += len(groups)
df['fold'] = df.group.map(group_fold)

test_files = {s: sorted(f for f in os.listdir(f'{DATA}/{s}') if f.endswith('.mp4')) for s in ['public_test', 'private_test']}
file_counts = np.bincount(df.label_idx, minlength=NUM_CLASSES)
LOG_PRIOR = np.log(file_counts / file_counts.sum()).astype(np.float32)

tr_df = df if VAL_FOLD is None else df[df.fold != VAL_FOLD]
va_df = df.iloc[:0] if VAL_FOLD is None else df[(df.fold == VAL_FOLD) & df.is_orig]
print(f'train files={len(tr_df)} groups={tr_df.group.nunique()} | val originals={len(va_df)} | '
      f'public={len(test_files["public_test"])} private={len(test_files["private_test"])}')

## Video reading, blank-frame removal, augmentation

In [ ]:
def read_video(path, size=224):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        if f.shape[:2] != (size, size):
            f = cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA)
        frames.append(f)
    cap.release()
    if not frames:
        raise ValueError(f'Could not read any frames from {path}')
    return np.stack(frames)


def drop_blank(frames, min_std=15, min_keep=4):
    """Remove flat single-colour padding frames (start/end of ~20% of train videos)."""
    std = frames.reshape(len(frames), -1).std(1)
    keep = std >= min_std
    return frames[keep] if keep.sum() >= min_keep else frames


def sample_idx(n, t=T, phase=0.5, train=False):
    """One frame per equal segment: random position when training, fixed phase otherwise."""
    if train and n > t:
        # random temporal window covering 80-100% of the clip (speed jitter)
        w = max(t, int(round(n * np.random.uniform(0.8, 1.0))))
        s = np.random.randint(0, n - w + 1)
    else:
        w, s = n, 0
    edges = np.linspace(0, w, t + 1)
    pos = np.random.rand(t) if train else np.full(t, phase)
    idx = s + edges[:-1] + pos * (edges[1:] - edges[:-1])
    return np.clip(idx.astype(int), 0, n - 1)


def augment_clip(frames):
    """Same random transform for every frame of the clip. Geometry only zooms out, so nothing near the borders is lost."""
    n, h, w, _ = frames.shape
    s = np.random.uniform(0.75, 1.0)
    ang = np.random.uniform(-10, 10)
    tx, ty = np.random.uniform(-1, 1, 2) * (1 - s) * np.array([w, h]) / 2
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, s)
    M[:, 2] += (tx, ty)
    fill = tuple(int(v) for v in np.random.randint(0, 256, 3)) if np.random.rand() < 0.5 \
        else tuple(int(v) for v in frames[0].reshape(-1, 3).mean(0))
    out = np.stack([cv2.warpAffine(f, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=fill) for f in frames])

    # colour: hue / saturation / brightness / contrast (signers wear different clothes, rooms differ)
    hsv = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2HSV) for f in out]).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + np.random.uniform(-12, 12)) % 180
    hsv[..., 1] *= np.random.uniform(0.6, 1.4)
    hsv[..., 2] *= np.random.uniform(0.7, 1.3)
    out = np.stack([cv2.cvtColor(f, cv2.COLOR_HSV2RGB) for f in np.clip(hsv, 0, 255).astype(np.uint8)]).astype(np.float32)
    c = np.random.uniform(0.7, 1.3)
    out = (out - out.mean()) * c + out.mean()
    if np.random.rand() < 0.15:
        out[:] = out.mean(-1, keepdims=True)
    if np.random.rand() < 0.15:
        out = out[..., np.random.permutation(3)]

    # degradations seen in the test sets (~5% heavily noisy videos)
    r = np.random.rand()
    if r < 0.25:
        out += np.random.normal(0, np.random.uniform(3, 30), out.shape)
    elif r < 0.35:
        mask = np.random.rand(n, h, w) < np.random.uniform(0.01, 0.08)
        out[mask] = np.random.choice([0, 255], size=(mask.sum(), 1))
    if np.random.rand() < 0.15:
        k = int(np.random.choice([3, 5]))
        out = np.stack([cv2.GaussianBlur(f, (k, k), 0) for f in out])
    return np.clip(out, 0, 255).astype(np.uint8)


MEAN = torch.tensor(WEIGHTS.transforms().mean).view(1, 3, 1, 1)
STD = torch.tensor(WEIGHTS.transforms().std).view(1, 3, 1, 1)

def to_tensor(frames):
    x = torch.from_numpy(frames).permute(0, 3, 1, 2).float().div_(255)
    return (x - MEAN) / STD


class VideoDataset(Dataset):
    def __init__(self, paths, labels=None, train=False, phase=0.5):
        self.paths, self.labels, self.train, self.phase = list(paths), labels, train, phase

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        frames = drop_blank(read_video(self.paths[i]))
        frames = frames[sample_idx(len(frames), T, phase=self.phase, train=self.train)]
        if self.train:
            frames = augment_clip(frames)
        return to_tensor(frames), (-1 if self.labels is None else int(self.labels[i]))


def epoch_paths(frame_df):
    """One random file per original video each epoch: copies act as extra colour augmentation, not extra weight."""
    pick = frame_df.groupby('group').sample(1)
    return pick.path.values, pick.label_idx.values

In [ ]:
# Look at a few augmented clips before spending GPU hours
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
for r in range(3):
    p = df.path.iloc[np.random.randint(len(df))]
    fr = drop_blank(read_video(p))
    fr = augment_clip(fr[sample_idx(len(fr), T, train=True)])
    for c in range(6):
        axes[r, c].imshow(fr[c * 3]); axes[r, c].axis('off')
plt.tight_layout(); plt.show()

## Model

In [ ]:
class TemporalHead(nn.Module):
    """Frame features (B,T,in_dim) -> CLS-token Transformer over time -> logits."""
    def __init__(self, in_dim=2048, d=384, depth=2, heads=6, drop=0.2, n_cls=NUM_CLASSES, t_max=32):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(drop), nn.Linear(in_dim, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.zeros(1, t_max + 1, d))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.fc = nn.Linear(d, n_cls)

    def forward(self, x):
        x = self.inp(x)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], dim=1) + self.pos[:, :x.shape[1] + 1]
        return self.fc(self.norm(self.enc(x)[:, 0]))


class ConvNeXtVideo(nn.Module):
    def __init__(self, weights=WEIGHTS):
        super().__init__()
        cnn = convnext_base(weights=weights, stochastic_depth_prob=DROP_PATH)
        self.features = cnn.features   # [stem, stage1, down, stage2, down, stage3(27), down, stage4]
        self.norm = cnn.classifier[0]  # pretrained LayerNorm2d on the final 1024x7x7 map
        self.head = TemporalHead()

    def forward(self, x):  # (B,T,3,224,224)
        B, Tn = x.shape[:2]
        z = x.flatten(0, 1)
        for m in self.features:
            if isinstance(m, nn.Sequential) and self.training and GRAD_CKPT and not isinstance(m[0], (nn.Conv2d, nn.LayerNorm)):
                for blk in m:
                    z = checkpoint(blk, z, use_reentrant=False)
            else:
                z = m(z)
        z = self.norm(z)
        f = torch.cat([z.mean((2, 3)), z.amax((2, 3))], 1)
        return self.head(f.view(B, Tn, -1).float())


def layer_id(name, top=12):
    """Depth group of a backbone parameter: stem 0, stage1 1, stage2 2, stage3 3..11 (3 blocks each), stage4/norm 12."""
    if name.startswith('norm.'):
        return top
    i = int(name.split('.')[1])
    if i == 0:
        return 0
    if i in (1, 2, 3):
        return 1 if i == 1 else 2
    if i == 4:
        return 3
    if i == 5:
        return 3 + int(name.split('.')[2]) // 3
    return top


def param_groups(model):
    """Layer-wise LR decay: the top depth group gets LR_BACKBONE, each group below x LAYER_DECAY."""
    by_id = {}
    for n, p in model.named_parameters():
        if not n.startswith('head.'):
            by_id.setdefault(layer_id(n), []).append(p)
    groups = [{'params': list(model.head.parameters()), 'lr': LR_HEAD}]
    for i, ps in sorted(by_id.items()):
        groups.append({'params': ps, 'lr': LR_BACKBONE * LAYER_DECAY ** (12 - i)})
    return groups


def cosine_with_warmup(opt, total, warmup):
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))


def macro_f1(y, logits):
    return f1_score(y, logits.argmax(1), average='macro', labels=np.arange(NUM_CLASSES), zero_division=0)


@torch.no_grad()
def predict(model, paths, phases=(0.5,), desc='predict'):
    """Mean logits over temporal TTA phases."""
    model.eval()
    total = 0
    for ph in phases:
        loader = DataLoader(VideoDataset(paths, phase=ph), batch_size=BATCH * 2, num_workers=NUM_WORKERS)
        out = []
        for x, _ in tqdm(loader, desc=f'{desc} φ={ph}', leave=False):
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=DEVICE == 'cuda'):
                out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
        total = total + torch.cat(out).numpy()
    return total / len(phases)

## Train (resumes from `<run>_last.pth` in /kaggle/working or any attached /kaggle/input)

In [ ]:
core = ConvNeXtVideo().to(DEVICE)
opt = torch.optim.AdamW(param_groups(core), weight_decay=WEIGHT_DECAY)
assert sum(len(g['params']) for g in opt.param_groups) == len(list(core.parameters()))
model = nn.DataParallel(core) if N_GPU > 1 else core   # checkpoints always store `core` (no 'module.' prefix)
steps_per_epoch = tr_df.group.nunique() // BATCH
sched = cosine_with_warmup(opt, EPOCHS * steps_per_epoch, steps_per_epoch)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda' and AMP_DTYPE == torch.float16)
crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

ckpt_path = f'{OUT_DIR}/{RUN_NAME}_last.pth'
resume = [p for p in [ckpt_path, *sorted(glob.glob(f'/kaggle/input/**/{RUN_NAME}_last.pth', recursive=True))] if os.path.exists(p)]
start_epoch, history = 0, []
if resume:
    ck = max((torch.load(p, map_location='cpu', weights_only=False) for p in resume), key=lambda c: c['epoch'])
    core.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler']); start_epoch, history = ck['epoch'], ck['history']
    del ck
    print(f'resumed from epoch {start_epoch}')

for ep in range(start_epoch, EPOCHS):
    if history and (time.time() - T_START) / 3600 + 1.5 * max(h['min'] for h in history) / 60 > TIME_BUDGET_H:
        print(f'time budget reached: stopping after epoch {ep}'); break
    seed_all(SEED + ep)
    paths, labels = epoch_paths(tr_df)
    loader = DataLoader(VideoDataset(paths, labels, train=True), batch_size=BATCH, shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE == 'cuda', persistent_workers=False)
    model.train()
    tot, t0 = 0.0, time.time()
    pbar = tqdm(loader, desc=f'epoch {ep + 1}/{EPOCHS}', mininterval=30)   # sparse updates keep the committed log small
    for step, (x, y) in enumerate(pbar, 1):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=DEVICE == 'cuda'):
            loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(core.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
        pbar.set_postfix(loss=f'{tot / step:.4f}')
    rec = {'epoch': ep + 1, 'loss': tot / len(loader), 'min': (time.time() - t0) / 60}
    if len(va_df):
        logits = predict(model, va_df.path.values, desc='val')
        rec['f1'] = macro_f1(va_df.label_idx.values, logits)
        rec['f1_la'] = macro_f1(va_df.label_idx.values, logits - 0.5 * LOG_PRIOR)
    history.append(rec)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in rec.items()}, flush=True)
    torch.save({'model': core.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': ep + 1, 'history': history}, ckpt_path)

## Predict test → `cv_submission_<run>.zip`

Uses the **last** epoch. τ for logit adjustment is tuned on val when there is one, else 0.25 (best τ of the ViT fold-0 run). Val logits are saved too, for choosing ensemble weights with the ViT later.

In [ ]:
TTA_PHASES = (0.25, 0.5, 0.75)

TAU = 0.25
if len(va_df):
    va_logits = predict(model, va_df.path.values, TTA_PHASES, desc='val tta')
    np.save(f'{OUT_DIR}/logits_{RUN_NAME}_val.npy', va_logits)
    va_df[['path', 'label_idx']].to_csv(f'{OUT_DIR}/val_{RUN_NAME}.csv', index=False)
    taus = np.arange(0, 1.51, 0.25)
    scores = [macro_f1(va_df.label_idx.values, va_logits - t * LOG_PRIOR) for t in taus]
    TAU = float(taus[int(np.argmax(scores))])
    print('val macro-F1 by τ:', dict(zip(taus.round(2), np.round(np.array(scores) * 100, 2))), '-> τ =', TAU)

for split in ['public_test', 'private_test']:
    logits = predict(model, [f'{DATA}/{split}/{f}' for f in test_files[split]], TTA_PHASES, desc=split)
    np.save(f'{OUT_DIR}/logits_{RUN_NAME}_{split}.npy', logits)   # raw logits, for ensembling with the ViT
    adj = F.log_softmax(torch.from_numpy(logits), 1).numpy() - TAU * LOG_PRIOR
    sub = pd.DataFrame({'video_name': test_files[split], 'label': [idx_to_label[int(p)] for p in adj.argmax(1)]})
    if split == 'private_test':
        csv_path = f'{OUT_DIR}/cv_submission.csv'
        sub.to_csv(csv_path, index=False, encoding='utf-8')
        zip_path = f'{OUT_DIR}/cv_submission_{RUN_NAME}.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(csv_path, arcname='cv_submission.csv')   # csv at the zip root, as required
        print(f'{zip_path}: {len(sub)} rows, {sub.label.nunique()} distinct labels, top {sub.label.value_counts().head(3).to_dict()}')

with open(f'{OUT_DIR}/history_{RUN_NAME}.json', 'w') as f:
    json.dump({'history': history, 'tau': TAU}, f)